In [0]:
%run ./00_config

### Synthetic Live Ride Request Generator
Generates JSONL files simulating real-time taxi ride requests.
Auto Loader in SDP Pipeline 2 continuously picks up new files from the landing zone.

In [0]:
import json, random, uuid, os, time
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

NY_TZ = ZoneInfo("America/New_York")

# Output path: Auto Loader watches this directory
LIVE_VOLUME_PATH = "/Volumes/nyc_taxi_company/bronze/landing_files/synthetic/live_ride_requests"
os.makedirs(LIVE_VOLUME_PATH, exist_ok=True)

# --- Tunable parameters ---
BATCH_SIZE = 100          # Base requests per file (scaled by hour-of-day)
INTERVAL_SECONDS = 10     # Pause between batches
NUM_BATCHES = 30          # Total batches to generate (-1 for infinite until interrupted)

print(f"Output path: {LIVE_VOLUME_PATH}")
print(f"Batch size: ~{BATCH_SIZE} (adjusted by hour)")
print(f"Interval: {INTERVAL_SECONDS}s | Batches: {NUM_BATCHES if NUM_BATCHES > 0 else 'infinite'}")

In [0]:
# Pickup zone weights derived from historical silver.trips_clean volume.
# Top 30 zones cover ~75% of all trips.
ZONE_WEIGHTS = {
    # Manhattan (dominant)
    161: 0.048, 237: 0.047, 236: 0.044, 162: 0.036, 230: 0.035,
    186: 0.034, 170: 0.033, 234: 0.030, 163: 0.029, 164: 0.029,
    239: 0.027, 68: 0.027, 48: 0.026, 79: 0.024, 158: 0.024,
    249: 0.024, 113: 0.023, 114: 0.021, 246: 0.019, 100: 0.018,
    # Queens (airports)
    132: 0.044, 138: 0.030,
    # Brooklyn
    189: 0.010, 97: 0.008, 65: 0.008, 181: 0.006,
    # Bronx
    247: 0.005, 69: 0.004, 119: 0.004,
    # Queens (non-airport)
    93: 0.005, 92: 0.004,
}

# Normalize to sum to 1.0
_total = sum(ZONE_WEIGHTS.values())
ZONE_WEIGHTS = {k: v / _total for k, v in ZONE_WEIGHTS.items()}
ZONE_IDS = list(ZONE_WEIGHTS.keys())
ZONE_PROBS = list(ZONE_WEIGHTS.values())

VEHICLE_TYPES = ["yellow", "green", "black_car", "shared"]
VEHICLE_PROBS = [0.60, 0.15, 0.15, 0.10]

# Hour-of-day demand multiplier (1.0 = peak)
HOUR_MULTIPLIERS = {
    0: 0.30,  1: 0.20,  2: 0.15,  3: 0.10,  4: 0.10,  5: 0.20,
    6: 0.50,  7: 0.80,  8: 1.00,  9: 0.90, 10: 0.70, 11: 0.70,
    12: 0.80, 13: 0.70, 14: 0.70, 15: 0.80, 16: 0.90, 17: 1.00,
    18: 1.00, 19: 0.90, 20: 0.80, 21: 0.70, 22: 0.50, 23: 0.40,
}

print(f"Zones: {len(ZONE_IDS)} | Vehicle types: {len(VEHICLE_TYPES)}")

In [0]:
def generate_request(event_ts):
    """Generate a single synthetic ride request with realistic distributions."""
    pickup = random.choices(ZONE_IDS, weights=ZONE_PROBS, k=1)[0]
    dropoff = random.choices(ZONE_IDS, weights=ZONE_PROBS, k=1)[0]
    while dropoff == pickup:
        dropoff = random.choices(ZONE_IDS, weights=ZONE_PROBS, k=1)[0]

    distance = round(max(0.3, random.gauss(4.0, 3.0)), 2)
    base_fare = 2.50 + distance * 2.50 + random.uniform(0, 5)

    # Peak-hour surge pricing
    hour = event_ts.hour
    if hour in range(7, 10) or hour in range(17, 20):
        base_fare *= random.uniform(1.1, 1.5)

    return {
        "request_id": str(uuid.uuid4()),
        "event_ts": event_ts.strftime("%Y-%m-%dT%H:%M:%S"),
        "pickup_location_id": pickup,
        "dropoff_location_id": dropoff,
        "passenger_count": random.choices([1, 2, 3, 4], weights=[0.6, 0.2, 0.1, 0.1], k=1)[0],
        "requested_vehicle_type": random.choices(VEHICLE_TYPES, weights=VEHICLE_PROBS, k=1)[0],
        "estimated_distance_miles": distance,
        "estimated_fare_amount": round(base_fare, 2),
        "request_status": "requested",
        "schema_version": "1.0",
        "_generator_scenario": "baseline",
    }


def generate_batch(batch_size, base_ts):
    """Generate a batch of requests, size scaled by hour-of-day multiplier."""
    multiplier = HOUR_MULTIPLIERS.get(base_ts.hour, 0.5)
    adjusted_size = max(10, int(batch_size * multiplier))
    return [
        generate_request(base_ts + timedelta(seconds=random.uniform(0, INTERVAL_SECONDS)))
        for _ in range(adjusted_size)
    ]


def write_batch(requests, batch_num):
    """Write requests as JSONL to the landing zone. Returns (filename, count)."""
    tag = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"batch_{tag}_{batch_num:04d}.json"
    filepath = os.path.join(LIVE_VOLUME_PATH, filename)
    with open(filepath, "w") as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")
    return filename, len(requests)


print("Generator functions ready")

In [0]:
# --- Main generation loop ---
# Each iteration writes one JSONL file to the landing zone.
# Auto Loader (SDP Pipeline 2) picks up new files continuously.
# Interrupt the cell (cancel) to stop generation.

print(f"Starting generator: {NUM_BATCHES} batches, ~{BATCH_SIZE} requests each\n")

total_requests = 0
batch_num = 0

try:
    while NUM_BATCHES < 0 or batch_num < NUM_BATCHES:
        now = datetime.now(NY_TZ)
        requests = generate_batch(BATCH_SIZE, now)
        filename, count = write_batch(requests, batch_num)

        total_requests += count
        batch_num += 1
        print(f"[{batch_num:>4}] {filename}: {count:>4} requests  (cumulative: {total_requests:,})")

        if NUM_BATCHES < 0 or batch_num < NUM_BATCHES:
            time.sleep(INTERVAL_SECONDS)

except KeyboardInterrupt:
    print(f"\nInterrupted after {batch_num} batches")

print(f"\nDone: {batch_num} batches, {total_requests:,} total requests")

In [0]:
# Quick validation: list files and sample the latest one

files = sorted([f for f in os.listdir(LIVE_VOLUME_PATH) if f.endswith(".json")])
print(f"Files in landing zone: {len(files)}")

if files:
    latest = files[-1]
    sample_path = f"{LANDING_PATH}/synthetic/live_ride_requests/{latest}"
    sample_df = spark.read.json(sample_path)
    print(f"Latest file: {latest} ({sample_df.count()} records)")
    display(sample_df.limit(5))